# Odd Lots: from transcripts to stock-call candidates

1. Three models independently read each locally obtained episode transcript.
2. Company/direction agreement is checked with one vote per model.
3. A strict judge checks current, explicit long/short investment views in context.
4. Ticker proposals are checked against a cached current SEC/Yahoo identity lookup.

Run from the repository or its `notebooks/` folder. Local transcripts, raw responses,
and intermediate files stay under `private/` (`ODD_LOTS_PRIVATE_DIR` overrides it).
Network calls are disabled by default: set `ODD_LOTS_ALLOW_NETWORK=1` deliberately to
fetch missing records, and supply `OPENROUTER_API_KEY` through the environment.
Cached failures remain errors; `ODD_LOTS_RETRY_ERRORS=1` explicitly retries them.

**This is a retrospective classification, not a point-in-time investment signal.**
Today's model knowledge and today's security lookup can contain future information.
An unresolved historical/delisted security remains in the ledger for review; a current
security lookup does not prove listing eligibility on the episode date.

## 1. The data

In [ ]:
import os, sys, json, re, time, random, hashlib, datetime as dt
from collections import Counter
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import duckdb, polars as pl, pandas as pd, requests
from rapidfuzz import fuzz, process

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
assert (ROOT / "scripts" / "extraction_utils.py").exists(), "Run from the repository root or notebooks/"
sys.path.insert(0, str(ROOT / "scripts"))
from extraction_utils import validate_groups, canonical_ticker, company_matches, dated_ticker_override, request_hash, require_stage_success
PRIVATE = Path(os.environ.get("ODD_LOTS_PRIVATE_DIR") or "private").expanduser()
PRIVATE = (PRIVATE if PRIVATE.is_absolute() else ROOT / PRIVATE).resolve()
PARQUET, OUT = PRIVATE / "parquet", PRIVATE / "llm_extractions"
CACHE = OUT / "cache"; CACHE.mkdir(parents=True, exist_ok=True)
assert PARQUET.exists(), "Build local transcripts first (see README); they are not distributed"
ALLOW_NETWORK = os.environ.get("ODD_LOTS_ALLOW_NETWORK", "0") == "1"
RETRY_ERRORS = ALLOW_NETWORK and os.environ.get("ODD_LOTS_RETRY_ERRORS", "0") == "1"
API_KEY = os.environ.get("OPENROUTER_API_KEY")   # no personal key-file or automatic .env loader
pl.Config.set_tbl_rows(40); pl.Config.set_tbl_cols(20); pl.Config.set_tbl_width_chars(200)
pd.set_option("display.max_columns", None, "display.width", 250, "display.max_colwidth", 70)

con = duckdb.connect()
docs = con.sql(f"SELECT * EXCLUDE (text) FROM read_parquet('{PARQUET}/docs/**/*.parquet', hive_partitioning=true)").pl().sort("date")
if "token_count_o200k" not in docs.columns and "token_count" in docs.columns:
    docs = docs.rename({"token_count": "token_count_o200k"})  # legacy private corpus, renamed on the next build
print(f"{len(docs)} transcripts | {docs['episode_id'].n_unique()} distinct episodes | {docs['date'].min()} to {docs['date'].max()}")

In [ ]:
# how many, and from where
print(docs.group_by("source").agg(
    pl.len().alias("transcripts"), pl.col("date").min().alias("first"), pl.col("date").max().alias("last"),
    pl.col("has_speakers").first().alias("speaker_labels")))
print(docs.group_by("year", "source").len().pivot(on="source", index="year", values="len").sort("year").fill_null(0))

In [ ]:
# how long is each transcript?
print(docs.select("word_count", "token_count_o200k", "duration_s", "n_chunks").describe())
print(docs.group_by("source").agg(pl.col("word_count").mean().round().alias("mean_words"), pl.col("word_count").min().alias("min_words"),
                                  pl.col("word_count").max().alias("max_words"), (pl.col("duration_s").mean() / 60).round().alias("mean_minutes")))
print("total words:", docs["word_count"].sum(), "| total tokens:", docs["token_count_o200k"].sum())
print(docs.select("date", "source", "title", "word_count", "duration_s").sort("word_count", descending=True).head(3))
print(docs.select("date", "source", "title", "word_count", "duration_s").sort("word_count").head(3))

In [ ]:
# chunks: ~1-minute paragraphs (YouTube) or speaker turns (Apple)
print(con.sql(f"""SELECT source, count(*) AS chunks, count(DISTINCT doc_id) AS transcripts, round(avg(word_count)) AS mean_words_per_chunk,
                  count(DISTINCT speaker) AS speakers FROM read_parquet('{PARQUET}/chunks/**/*.parquet', hive_partitioning=true)
                  GROUP BY source""").pl())

## 2. Raw extraction: 3 models read every episode

In [ ]:
MODELS = [   # llm1, llm2, llm3
    {"id": "openai/gpt-5-nano",            "reasoning": {"effort": "minimal"}, "temperature": None},
    {"id": "deepseek/deepseek-v4.1-flash", "reasoning": {"enabled": False},   "temperature": 0},
    {"id": "google/gemini-2.5-flash-lite", "reasoning": {"enabled": False},   "temperature": 0},
]
for i, m in enumerate(MODELS, 1): m["name"] = f"llm{i}"
NAME2ID = {m["name"]: m["id"] for m in MODELS}
CONCURRENCY = 12
RUN_DATE = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")

# USD per million tokens (in, out). Snapshot of 2026-10-31, refreshed from the live OpenRouter catalogue only when a call has to be made.
PRICE = {"openai/gpt-5-nano": (0.05, 0.40), "deepseek/deepseek-v4.1-flash": (0.045, 0.50), "google/gemini-2.5-flash-lite": (0.10, 0.40)}
def refresh_prices(ids):
    if not ALLOW_NETWORK: raise RuntimeError("Network is disabled")
    try:
        cat = {m["id"]: m["pricing"] for m in requests.get("https://openrouter.ai/api/v1/models", timeout=60).json()["data"]}
        for i in ids: PRICE[i] = (float(cat[i]["pricing"]["prompt"] if "pricing" in cat[i] else cat[i]["prompt"]) * 1e6, float(cat[i]["completion"]) * 1e6)
    except Exception: print("live prices unavailable, using the snapshot")
    print(pl.DataFrame([{"model": i, "in_$/M": PRICE[i][0], "out_$/M": PRICE[i][1]} for i in ids]))

In [ ]:
SYSTEM = """You analyse transcripts of the Bloomberg "Odd Lots" podcast (hosts Joe Weisenthal and Tracy Alloway, plus a guest).

Question: what stock recommendation is given in the podcast? If there is none, return an empty list. If several different companies get a recommendation, return one item per company; do not stop at the first.

An item counts when a host or guest expresses a view on one specific, named company's listed stock as an investment, for example: "I'd buy X", "I'm long X", "short X", "I'd avoid X", "I own X", "X is cheap / overvalued", "X is my favourite stock", "X is the best way to play this", "X looks like a loser here".
Do NOT count: general macro, sector, country, index, ETF, crypto, commodity or bond views; neutral descriptions of what a company does; mere reports that a stock moved or what a company reported; other people's views mentioned in passing; jokes; pure hypotheticals.
Many episodes contain no stock view at all, and then the list is empty. That is the normal, expected answer.

For each recommendation return:
- company: correct company name (the transcript may be mis-heard auto-captions)
- ticker: if said or obvious, else ""
- direction: "bullish" (buy/long/overweight), "bearish" (sell/short/underweight/avoid) or "neutral"
- quote: words copied EXACTLY and consecutively from the transcript, at most 40 words, no speaker labels, no spelling fixes
- confidence: integer 0-10, how clearly this is an actual explicit stock recommendation

Also return date_recorded: the date the episode was recorded as YYYY-MM-DD, only if the transcript states or clearly implies it
(e.g. "we're recording this on Tuesday, February 7th"; the publish date is given), otherwise "".

Reply with one JSON object: {"date_recorded": "...", "recommendations": [ ... ]} and nothing else."""

SCHEMA = {"name": "stock_recs", "strict": True, "schema": {
    "type": "object", "additionalProperties": False, "required": ["date_recorded", "recommendations"],
    "properties": {"date_recorded": {"type": "string"}, "recommendations": {"type": "array", "items": {
        "type": "object", "additionalProperties": False, "required": ["company", "ticker", "direction", "quote", "confidence"],
        "properties": {"company": {"type": "string"}, "ticker": {"type": "string"},
                       "direction": {"type": "string", "enum": ["bullish", "bearish", "neutral"]},
                       "quote": {"type": "string"}, "confidence": {"type": "integer"}}}}}}}
PROMPT_HASH = hashlib.md5((SYSTEM + json.dumps(SCHEMA)).encode()).hexdigest()[:8]

# one transcript per episode: Apple (punctuation + speaker labels) unless it is clearly incomplete (<90% of the YouTube word count)
yt = docs.filter(pl.col("source") == "youtube_auto_captions").select("episode_id", yt_words="word_count")
eligible_docs = docs.filter(~pl.col("is_truncated").fill_null(False)) if "is_truncated" in docs.columns else docs
episodes = eligible_docs.join(yt, on="episode_id", how="left").filter(
    (pl.col("source") == "youtube_auto_captions") | pl.col("yt_words").is_null() | (pl.col("word_count") >= 0.9 * pl.col("yt_words")))
apple_ok = episodes.filter(pl.col("source") == "apple_podcasts_ttml")["episode_id"]
episodes = episodes.filter(~((pl.col("source") == "youtube_auto_captions") & pl.col("episode_id").is_in(apple_ok))).sort("date")
print(f"{len(docs)} transcripts -> {len(episodes)} episodes to process")
sel = episodes

ids = ",".join(f"'{i}'" for i in sel["doc_id"])
chunks = con.sql(f"SELECT doc_id, start_s, speaker, text FROM read_parquet('{PARQUET}/chunks/**/*.parquet', hive_partitioning=true) "
                 f"WHERE doc_id IN ({ids}) ORDER BY doc_id, chunk_idx").pl()
def norm(s): return re.sub(r"\W+", " ", (s or "").lower()).strip()      # lowercase + strip punctuation before any matching
CHUNKS = {k[0]: [(st, sp, tx, norm(tx)) for st, sp, tx in zip(g["start_s"], g["speaker"], g["text"])]     # (start_s, speaker, text, normalised)
          for k, g in chunks.group_by("doc_id", maintain_order=True)}
def lines(cs): return "\n\n".join(f"{sp}: {tx}" if sp else tx for _, sp, tx, _ in cs)   # Apple turns get "SPEAKER_n:" labels
def user_msg(d): return f"Episode: {d['title']} (published {d['date']})\n\nTRANSCRIPT:\n{lines(CHUNKS[d['doc_id']])}"

In [ ]:
URL = "https://openrouter.ai/api/v1/chat/completions"
slug = lambda m: m.replace("/", "__")
COSTS = []     # (stage, model, cost) for every call used in this notebook, cached or not
GENERATED = [] # successful model generations in this run; cached evidence is counted separately

def build_body(model, system, user, schema):
    body = {"model": model["id"], "max_tokens": model.get("max_tokens", 8000), "usage": {"include": True},
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
            "response_format": {"type": "json_schema", "json_schema": schema},     # structured output
            "provider": {"require_parameters": True, "sort": "price"},             # cheapest provider that supports it
            "reasoning": model["reasoning"]}
    if model["temperature"] is not None: body["temperature"] = model["temperature"]
    return body
def cache_path(body, stage):    # the key covers the whole request: model, prompt, schema, input, temperature, max_tokens, provider
    return CACHE / slug(body["model"]) / f"{stage}_{hashlib.md5(json.dumps(body, sort_keys=True).encode()).hexdigest()[:20]}.json"
def is_cached(model, system, user, schema, stage): return cache_path(build_body(model, system, user, schema), stage).exists()

def call_llm(model, system, user, schema, stage):
    """Replay a full-request cache; fetching and retrying cached errors require explicit opt-in."""
    body = build_body(model, system, user, schema); path = cache_path(body, stage)
    if path.exists():
        out = json.loads(path.read_text())
        if out.get("status", "ok") != "error":
            COSTS.append((stage, model["id"], out.get("cost", 0)))
            return out
        if not RETRY_ERRORS: raise RuntimeError("Cached failure: " + out["error"])
    if not ALLOW_NETWORK: raise RuntimeError(f"Offline cache miss: {stage}/{path.name}")
    if not API_KEY: raise RuntimeError("OPENROUTER_API_KEY is not set")
    path.parent.mkdir(parents=True, exist_ok=True)
    last = ""
    for attempt in range(5):
        try:
            r = requests.post(URL, headers={"Authorization": "Bearer " + API_KEY}, json=body, timeout=180)
            j = r.json()
            if r.status_code == 200 and "error" not in j:
                choice = j["choices"][0]; text = choice["message"].get("content") or ""
                if choice.get("finish_reason") == "length": raise ValueError("truncated response")
                data = json.loads(text[text.find("{"): text.rfind("}") + 1])
                out = {"status": "ok", "data": data, "content": text, "response": j,
                       "provider": j.get("provider"), "cost": j.get("usage", {}).get("cost", 0),
                       "created": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
                       "model": model["id"], "stage": stage, "request_hash": request_hash(body),
                       "request_parameters": {k: v for k, v in body.items() if k != "messages"},
                       "prompt_hash": request_hash({"system": system, "schema": schema}),
                       "tokens_in": j.get("usage", {}).get("prompt_tokens"),
                       "tokens_out": j.get("usage", {}).get("completion_tokens")}
                path.write_text(json.dumps(out))
                GENERATED.append((stage, model["id"]))
                COSTS.append((stage, model["id"], out["cost"]))
                return out
            last = f"HTTP {r.status_code}" if r.status_code != 200 else "provider error body"
            if 400 <= r.status_code < 500 and r.status_code not in (408, 429): break
        except (requests.RequestException, ValueError, KeyError, IndexError, TypeError) as exc:
            last = type(exc).__name__       # never persist a raw HTTP exception containing a credential
        if attempt < 4: time.sleep(2 ** attempt + random.random())
    failure = {"status": "error", "error": last, "model": model["id"], "stage": stage,
               "request_hash": request_hash(body), "created": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")}
    path.write_text(json.dumps(failure))
    raise RuntimeError(f"{stage} failed after bounded retries ({last})")

def run_parallel(fn, jobs):
    """One pass per job; call_llm owns bounded retries. Cached errors need explicit retry opt-in."""
    def safe(job):
        try: return job, fn(job), None
        except Exception as e: return job, None, f"{type(e).__name__}: {e}"[:200]
    with ThreadPoolExecutor(CONCURRENCY) as ex: res = list(ex.map(safe, jobs))
    return res

jobs = [(m, d) for d in sel.to_dicts() for m in MODELS]
ask = lambda job: call_llm(job[0], SYSTEM, user_msg(job[1]), SCHEMA, "extract")        # stage 1: one model reads one episode
pending = [(m, d) for m, d in jobs if not is_cached(m, SYSTEM, user_msg(d), SCHEMA, "extract")]
print(f"{len(jobs)} calls, {len(pending)} not cached yet")
if pending and ALLOW_NETWORK:
    refresh_prices([m["id"] for m in MODELS])
    est = sum(((d["token_count_o200k"] + 600) * PRICE[m["id"]][0] + 300 * PRICE[m["id"]][1]) / 1e6 for m, d in pending)
    print(f"estimated at most ${est:.4f}"); assert est < 3, "estimate above $3: check episode selection"
t0 = time.time()
results = run_parallel(ask, jobs)           # all model x episode calls in parallel
errors = [(m["id"], d["doc_id"], e) for (m, d), _, e in results if e]
print(f"done in {time.time() - t0:.0f}s, {len(errors)} errors")
for e in errors: print("  ERROR", e)

In [ ]:
# one row per podcast (index = podcast date); columns per model: date_recorded, trade_reco, quote, confidence, quote_score, error
FULL = {did: norm(" ".join(c[3] for c in cs)) for did, cs in CHUNKS.items()}
def locate(doc_id, quote):     # (chunk index, start_s) of the chunk that best matches the quote
    q = norm(quote); sc = [fuzz.partial_ratio(q, c[3]) for c in CHUNKS[doc_id]]; i = max(range(len(sc)), key=sc.__getitem__)
    return i, float(CHUNKS[doc_id][i][0])
FIELDS = ["date_recorded", "trade_reco", "quote", "confidence", "quote_score", "error"]

rows, long, runs = {}, [], []     # long = one row per model x recommendation (used for cleaning); runs = one row per model x episode
for (m, d), res, err in results:
    n = m["name"]; row = rows.setdefault(d["doc_id"], {"date": d["date"], "doc_id": d["doc_id"], "title": d["title"], "source": d["source"]})
    runs.append({"llm": n, "model": m["id"], "doc_id": d["doc_id"], "status": "error" if err else "ok", "error": err,
                 "n_recs": None, "cost_usd": res["cost"] if res else None, "provider": res.get("provider") if res else None,
                 "request_hash": request_hash(build_body(m, SYSTEM, user_msg(d), SCHEMA)),
                 "cached_response_created": res.get("created") if res else None,
                 "run_date": RUN_DATE})
    if res is None: row[f"error_{n}"] = err; continue          # an error is NOT "no recommendation": the fields stay empty and the error is shown
    recs = [r for r in res["data"]["recommendations"] if r["company"].strip()]
    for r in recs: r["quote"] = " ".join(r["quote"].split()[:40])        # enforce the 40-word limit
    runs[-1]["n_recs"] = len(recs)
    row[f"date_recorded_{n}"] = res["data"]["date_recorded"] or None
    row[f"trade_reco_{n}"] = " | ".join(f"{r['company']}{' (' + r['ticker'] + ')' if r['ticker'] else ''} {r['direction']}" for r in recs)
    row[f"quote_{n}"] = " | ".join(r["quote"] for r in recs)
    row[f"confidence_{n}"] = " | ".join(str(r["confidence"]) for r in recs)
    scores = [fuzz.partial_ratio(norm(r["quote"]), FULL[d["doc_id"]]) for r in recs]    # 100 = verbatim, <90 = probably invented
    row[f"quote_score_{n}"] = min(scores, default=None)
    for r, sc in zip(recs, scores):
        ci, st = locate(d["doc_id"], r["quote"])
        long.append({"doc_id": d["doc_id"], "date": d["date"], "title": d["title"], "source": d["source"], "llm": n,
                     "date_recorded": res["data"]["date_recorded"] or None, **r, "quote_score": sc, "chunk_idx": ci, "start_s": st if sc >= 90 else None})

cols = ["doc_id", "title", "source"] + [f"{f}_{m['name']}" for m in MODELS for f in FIELDS]
df = pd.DataFrame(rows.values()).reindex(columns=["date"] + cols).set_index("date").sort_index()
df.to_parquet(OUT / "stock_recs_wide.parquet"); df.to_csv(OUT / "stock_recs_wide.csv")
pd.DataFrame(runs).to_parquet(OUT / "runs.parquet")
long = pd.DataFrame(long, columns=["doc_id", "date", "title", "source", "llm", "date_recorded", "company", "ticker", "direction", "quote",
                                   "confidence", "quote_score", "chunk_idx", "start_s"])
print({m["name"]: m["id"] for m in MODELS})
df.filter(like="trade_reco").ne("").any(axis=1).sum()
require_stage_success("extraction", errors)

In [ ]:
# raw performance of each model (before any cleaning)
r_ = pd.DataFrame(runs)
perf = pd.DataFrame({m["name"]: {
    "model": m["id"],
    "episodes_ok": (r_[r_.llm == m["name"]].status == "ok").sum(),
    "episodes_error": (r_[r_.llm == m["name"]].status == "error").sum(),
    "episodes_with_reco": long[long.llm == m["name"]].doc_id.nunique(),
    "recommendations": (long.llm == m["name"]).sum(),
    "quotes_verbatim (score>=90)": f"{(long[long.llm == m['name']].quote_score >= 90).sum()}/{(long.llm == m['name']).sum()}",
    "avg_confidence": round(long[long.llm == m["name"]].confidence.mean(), 1) if (long.llm == m["name"]).any() else None,
    "episodes_with_date_recorded": df[f"date_recorded_{m['name']}"].notna().sum()} for m in MODELS}).T
print(perf.to_string())
n_models = long.groupby("doc_id").llm.nunique()
print(f"\nepisodes with a recommendation from >=1 model: {(n_models >= 1).sum()} | >=2 models: {(n_models >= 2).sum()} | all 3: {(n_models >= 3).sum()}  (of {len(df)} episodes)")

## 3. Cleaning: keep only what 2 of 3 models agree on (company + direction)

In [ ]:
# An LLM only groups the different spellings of the same company ("Nvidia Corp", "NVIDIA", "NVDA").
# The 2-of-3 rule on the direction is applied in code. Only episodes where >=2 models found something need a call.
MATCH_MODEL = MODELS[1]
MATCH_SYSTEM = """You are given numbered mentions of companies extracted from one podcast episode by different analysts.
Group the numbers that refer to the same company (same listed company, ignoring spelling, casing, legal suffixes, tickers vs names).
Different companies must stay in different groups, even if they are related (e.g. Alphabet vs Google is ONE company; Ford vs GM are TWO).
Every number must appear in exactly one group. For each group give the clean, common company name.
Reply with one JSON object {"groups": [{"company": "...", "items": [1, 2]}]} and nothing else."""
MATCH_SCHEMA = {"name": "company_groups", "strict": True, "schema": {"type": "object", "additionalProperties": False, "required": ["groups"],
    "properties": {"groups": {"type": "array", "items": {"type": "object", "additionalProperties": False, "required": ["company", "items"],
        "properties": {"company": {"type": "string"}, "items": {"type": "array", "items": {"type": "integer"}}}}}}}}

def good_date(values, published):     # most common stated recording date, only if plausible (0-60 days before publication)
    ok = []
    for v in values:
        try: d = pd.Timestamp(v).date()
        except Exception: continue
        if 0 <= (pd.Timestamp(published).date() - d).days <= 60: ok.append(str(d))
    return sorted(Counter(ok).items(), key=lambda kv: (-kv[1], kv[0]))[0][0] if ok else None     # deterministic tie-break

groups_of = {}
todo = []
for doc_id, g in long.groupby("doc_id"):
    g = g.reset_index(drop=True)
    if g.llm.nunique() < 2: continue                                   # no agreement possible
    groups_of[doc_id] = g
    todo.append((doc_id, "\n".join(f"{i + 1}. {r.company}" + (f" ({r.ticker})" if r.ticker else "") for i, r in g.iterrows())))
def match_one(job):
    groups = call_llm(MATCH_MODEL, MATCH_SYSTEM, job[1], MATCH_SCHEMA, "match")["data"]["groups"]
    validate_groups(groups, len(groups_of[job[0]]))
    return groups
matched = run_parallel(match_one, todo)
match_errors = [(j[0], e) for j, _, e in matched if e]
pd.DataFrame([{"doc_id": j[0], "error": e, "status": "error" if e else "ok"}
              for j, _, e in matched]).to_csv(OUT / "match_runs.csv", index=False)
print(len(todo), "episodes grouped by the matcher,", len(match_errors), "matcher errors")
require_stage_success("matcher", match_errors)

kept = []
for (doc_id, _), groups, err in matched:
    g = groups_of[doc_id]
    for grp in groups:
        sub = g.loc[[i - 1 for i in grp["items"]]]
        for direction, s in sub.groupby("direction"):
            s = s.sort_values(["confidence", "quote_score"], ascending=False).drop_duplicates("llm")     # one item per model before averaging
            if len(s) < 2: continue                                    # <2 models agree on this direction -> dropped
            best = s.sort_values(["quote_score", "confidence"], ascending=False).iloc[0]
            kept.append({"date": best["date"], "date_recorded": good_date(g.date_recorded.dropna(), best["date"]),
                         "company": grp["company"], "direction": direction, "quote": best["quote"], "n_models": len(s),
                         "models": ", ".join(NAME2ID[x] for x in sorted(s.llm)), "avg_confidence": round(s.confidence.mean(), 1),
                         "quote_score": best["quote_score"], "start_s": best["start_s"],
                         "items": list(zip(s.quote, s.chunk_idx)), "doc_id": doc_id, "title": best["title"], "match_error": err})
CLEAN_COLS = ["date", "date_recorded", "company", "direction", "quote", "n_models", "models", "avg_confidence", "quote_score", "start_s", "items", "doc_id", "title", "match_error"]
clean = pd.DataFrame(kept, columns=CLEAN_COLS).sort_values(["date", "company"]).reset_index(drop=True)
print(f"{len(long)} raw recommendations -> {len(clean)} candidates (company + direction agreed by >=2 of 3 models)")

## 4. Judge: a stricter model re-reads each candidate in context

In [ ]:
JUDGE = {"id": "deepseek/deepseek-v4.1-flash", "reasoning": {"effort": "medium"}, "temperature": 0, "max_tokens": 12000}
PRICE.setdefault(JUDGE["id"], (0.045, 0.50))
JUDGE_SYSTEM = """You are a strict reviewer of stock-recommendation candidates taken from the Bloomberg "Odd Lots" podcast (hosts Joe Weisenthal and Tracy Alloway, plus a guest).
You get a company, the direction analysts assigned, the quote(s) they flagged, and the transcript excerpt around each quote.

Return is_call=true ONLY if ALL of these hold:
1. A speaker (host or guest) expresses their OWN bullish or bearish view or position on the stock: buy, long, own, overweight / sell, short, avoid, underweight, or "cheap" / "overvalued" meant as an investment view.
2. It is a present-day view, not a historical anecdote ("we bought in 2008", "in 1929", "was one of my biggest positions") and not a position described only in the past.
3. The target is ONE specific, publicly listed company's stock.

Return is_call=false for: neutral or hold views; commodities (cocoa, gold, silver); indices read as stocks (the Nasdaq index); ETFs and funds; private companies (including media or data firms such as Bloomberg); jokes; sector, industry or credit views; companies cited only as an example of an economic phenomenon (pricing power, wages, capex, supply chains) with no investment view; a host merely remarking that a stock has risen or fallen; business commentary or description without a view; third parties' views that the speaker does not adopt.
When in doubt, is_call=false.

direction: "long" or "short" when is_call is true, otherwise "none". listed_company: true if the named company's stock is publicly traded. reason: one short sentence.
Reply with one JSON object and nothing else."""
JUDGE_SCHEMA = {"name": "judge", "strict": True, "schema": {"type": "object", "additionalProperties": False,
    "required": ["is_call", "direction", "listed_company", "reason"],
    "properties": {"is_call": {"type": "boolean"}, "direction": {"type": "string", "enum": ["long", "short", "none"]},
                   "listed_company": {"type": "boolean"}, "reason": {"type": "string"}}}}

def judge_user(c):
    cs = CHUNKS[c["doc_id"]]; k = 8 if cs[0][1] else 2          # +-8 speaker turns (Apple) or +-2 one-minute paragraphs (YouTube)
    parts, last = [], -99
    for q, i in sorted(c["items"], key=lambda x: x[1]):
        if i - last <= k: continue                                  # excerpt already covered
        parts.append(lines(cs[max(0, i - k): i + k + 1])); last = i
    quotes = "\n".join(f'- "{q}"' for q, _ in c["items"])
    return (f"Episode: {c['title']} (published {c['date']})\nCompany: {c['company']}\nDirection assigned by analysts: {c['direction']}\n"
            f"Flagged quote(s):\n{quotes}\n\nTranscript excerpt(s):\n" + "\n\n[...]\n\n".join(parts))

cand = clean.to_dict("records")
to_judge = [c for c in cand if c["direction"] != "neutral"]        # neutral is never a call
judge_in = {id(c): judge_user(c) for c in to_judge}
est = sum(len(u.split()) * 1.4 * PRICE[JUDGE["id"]][0] / 1e6 + 1500 * PRICE[JUDGE["id"]][1] / 1e6 for u in judge_in.values())
print(f"{len(cand)} candidates, {len(to_judge)} non-neutral to judge, estimated ${est:.3f}"); assert est < 1.0, "judge estimate above the $1 cap"
verdicts = run_parallel(lambda c: call_llm(JUDGE, JUDGE_SYSTEM, judge_in[id(c)], JUDGE_SCHEMA, "judge")["data"], to_judge)
verdict = {id(c): (v, e) for c, v, e in verdicts}
for c in cand:
    v, e = verdict.get(id(c), (None, None))
    c["judge_error"] = e
    c["is_call"] = bool(v and v["is_call"] and v["listed_company"] and v["direction"] in ("long", "short"))
    c["listed_company"] = v["listed_company"] if v else None
    c["judge_direction"] = v["direction"] if v else "none"
    c["judge_reason"] = v["reason"] if v else (f"JUDGE ERROR: {e}" if e else "consensus direction is neutral")
cands = pd.DataFrame(cand, columns=CLEAN_COLS + ["judge_error", "is_call", "listed_company", "judge_direction", "judge_reason"])
cands["is_call"] = cands.is_call.astype(bool)
cands[["doc_id", "company", "judge_error", "is_call"]].to_csv(OUT / "judge_runs.csv", index=False)
require_stage_success("judge", cands.loc[cands.judge_error.notna(), "judge_error"])
print(f"judge errors: {cands.judge_error.notna().sum()} | candidates {len(cands)} -> approved {cands.is_call.sum()} (rejected {(~cands.is_call).sum()})")
print(cands.groupby(["is_call", "direction"]).size().unstack(fill_value=0))
approved = cands[cands.is_call].copy()
approved["direction_consensus"] = approved.direction
approved["direction"] = approved.judge_direction.map({"long": "bullish", "short": "bearish"})
approved[["date", "company", "direction", "direction_consensus", "judge_reason"]]

## 5. Ticker identity checks (current master, not historical listing eligibility)

Voting uses the full security symbol. ADRs, local listings, and distinct share classes
are not merged simply because they share a symbol prefix. A single validated proposal
is marked for review; two-model agreement is required for automatic resolution.
Historical securities absent from today's lookup remain in the approved-call ledger.

In [ ]:
TICK_SYSTEM = """You map a company named in a podcast to its stock ticker. You get the company name, a quote from the podcast and the podcast date.
Return the ticker of the company's primary common stock that was listed on a stock exchange on that date, in Yahoo Finance format
(e.g. NVDA, F, BRK-B, GOOGL, 7203.T, SAP.DE, 0700.HK). If several share classes exist pick the most widely traded one.
If the company was not publicly listed (private, subsidiary, unlisted) set listed=false and ticker="". Never guess a ticker you do not know: use listed=false.
Reply with one JSON object and nothing else."""
TICK_SCHEMA = {"name": "ticker", "strict": True, "schema": {"type": "object", "additionalProperties": False,
    "required": ["listed", "ticker", "exchange", "legal_name"],
    "properties": {"listed": {"type": "boolean"}, "ticker": {"type": "string"}, "exchange": {"type": "string"}, "legal_name": {"type": "string"}}}}

def find_ticker(job):
    m, r = job
    out = call_llm(m, TICK_SYSTEM, f"Company: {r['company']}\nPodcast date: {r['date']}\nQuote: {r['quote']}", TICK_SCHEMA, "ticker")["data"]
    return {"ticker": out["ticker"].strip().upper() if out["listed"] else "", "legal_name": out.get("legal_name", "")}

rows_ = approved.to_dict("records")
found = run_parallel(find_ticker, [(m, r) for r in rows_ for m in MODELS])
tk_err = Counter()
for (m, r), proposal, e in found:
    r[f"ticker_{m['name']}"] = proposal["ticker"] if proposal else None
    r[f"ticker_legal_name_{m['name']}"] = proposal["legal_name"] if proposal else ""
    if e: r.setdefault("ticker_errors", []).append(f"{m['name']}: {e}"); tk_err[m["name"]] += 1
print("ticker call errors:", dict(tk_err) or "none")
pd.DataFrame([{"doc_id": r["doc_id"], "company": r["company"], "model": m["id"],
               "ticker": proposal["ticker"] if proposal else None,
               "legal_name": proposal["legal_name"] if proposal else "", "error": e, "status": "error" if e else "ok"}
              for (m, r), proposal, e in found]).to_csv(OUT / "ticker_runs.csv", index=False)

In [ ]:
# Identity lookup only: this is a CURRENT master and cannot establish episode-date listing status.
SEC_UA = os.environ.get("SEC_USER_AGENT")
sec_file = CACHE / "sec_company_tickers_exchange.json"
if not sec_file.exists() and ALLOW_NETWORK and SEC_UA:
    resp = requests.get("https://www.sec.gov/files/company_tickers_exchange.json", headers={"User-Agent": SEC_UA}, timeout=60)
    resp.raise_for_status(); sec_file.write_text(resp.text)
SEC = {canonical_ticker(tk): (name, exch) for _, name, tk, exch in json.loads(sec_file.read_text())["data"]} if sec_file.exists() else {}
SEC_HASH = hashlib.sha256(sec_file.read_bytes()).hexdigest() if sec_file.exists() else None

def yahoo_equity(symbol):
    if not re.fullmatch(r"[A-Z0-9^.=-]+", symbol): return None, "invalid_symbol"
    p = CACHE / "yahoo" / f"{symbol}.json"
    if p.exists(): quotes = json.loads(p.read_text())
    elif not ALLOW_NETWORK: return None, "offline_cache_miss"
    else:
        try:
            resp = requests.get("https://query2.finance.yahoo.com/v1/finance/search", params={"q": symbol, "quotesCount": 5, "newsCount": 0},
                                headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
            resp.raise_for_status(); payload = resp.json()
            if "quotes" not in payload: return None, "invalid_lookup_response"
            quotes = payload["quotes"]
        except (requests.RequestException, ValueError): return None, "lookup_error"
        p.parent.mkdir(exist_ok=True); p.write_text(json.dumps(quotes))
    for q in quotes:
        if canonical_ticker(q.get("symbol", "")) == symbol and q.get("quoteType") == "EQUITY":
            return q.get("longname") or q.get("shortname"), None
    return None, "not_in_current_lookup"

def validate(t, company, legal_name=""):
    symbol = canonical_ticker(t)
    if not symbol: return None, "no_proposal"
    if symbol in SEC: name, source, error = SEC[symbol][0], "sec", None
    else:
        name, error = yahoo_equity(symbol); source = "yahoo"
    if not name: return None, error
    if not company_matches(company, name, legal_name): return None, "company_name_mismatch"
    lookup_file = sec_file if source == "sec" else CACHE / "yahoo" / f"{symbol}.json"
    return {"ticker": symbol, "name": name, "src": source, "lookup_hash": hashlib.sha256(lookup_file.read_bytes()).hexdigest()}, None

tk_cols = [f"ticker_{m['name']}" for m in MODELS]
for r in rows_:
    valid, issues = [], []
    proposals = [canonical_ticker(r.get(c) or "") for c in tk_cols if r.get(c)]
    proposal_counts = Counter(proposals)
    r["proposed_ticker"] = sorted(proposal_counts, key=lambda t: (-proposal_counts[t], t))[0] if proposals else None
    for c in tk_cols:
        v, error = validate(r.get(c) or "", r["company"], r.get(c.replace("ticker_", "ticker_legal_name_"), ""))
        if v: valid.append(v)
        elif error != "no_proposal": issues.append(f"{c}: {error}")
    votes = Counter(v["ticker"] for v in valid)
    r.update(ticker=None, ticker_name=None, ticker_src=None, ticker_votes=0, ticker_status="unresolved", ticker_lookup_hash=None)
    if votes:
        top = sorted(votes, key=lambda t: (-votes[t], t))[0]
        r["ticker_votes"] = votes[top]
        if votes[top] >= 2:
            best = next(v for v in valid if v["ticker"] == top)
            r.update(ticker=top, ticker_name=best["name"], ticker_src=best["src"], ticker_status=f"agreed ({votes[top]} of 3)", ticker_lookup_hash=best["lookup_hash"])
        else: r["ticker_status"] = "review_required_single_validated_proposal"
    r["listing_status_at_episode"] = "not_verified"
    r["ticker_override_source"] = r["ticker_override_effective_date"] = None
    r["ticker_override_from"] = r["ticker_override_to"] = None
    override = dated_ticker_override(r["company"], r["date"], r["proposed_ticker"])
    if override:
        verified, error = validate(override["to"], r["company"])
        if verified:
            r.update(ticker=verified["ticker"], ticker_name=verified["name"], ticker_src=verified["src"],
                     ticker_lookup_hash=verified["lookup_hash"], ticker_status="dated_issuer_mapping",
                     listing_status_at_episode="dated_issuer_notice", ticker_override_source=override["source"],
                     ticker_override_effective_date=override["effective_date"],
                     ticker_override_from=override["from"], ticker_override_to=override["to"])
            # ticker_votes still counts original exact-symbol model votes (one for CRH), not this correction.
        else: issues.append(f"dated_mapping_target: {error}")
    r["ticker_errors"] = json.dumps(r.get("ticker_errors", []) + issues)
    r["security_master_scope"] = "current_identity_only"
    r["security_master_hash"] = SEC_HASH
res_ = pd.DataFrame(rows_)
print(res_.ticker_status.value_counts().to_dict() if len(res_) else "no approved candidates")

In [ ]:
# Keep EVERY judge-approved call, including unresolved/historical securities.
# Downstream baskets must explicitly reject or resolve missing ticker/listing/price information.
pipeline = {"models": MODELS, "extract_system": SYSTEM, "extract_schema": SCHEMA,
            "match_model": MATCH_MODEL, "match_system": MATCH_SYSTEM, "match_schema": MATCH_SCHEMA,
            "judge_model": JUDGE, "judge_system": JUDGE_SYSTEM, "judge_schema": JUDGE_SCHEMA,
            "ticker_system": TICK_SYSTEM, "ticker_schema": TICK_SCHEMA,
            "request_template": {k: v for k, v in build_body(MODELS[0], "", "", SCHEMA).items() if k != "messages"}}
code_hash = request_hash({"notebook_code": [c["source"] for c in json.loads((ROOT / "notebooks" / "odd_lots_stock_recs.ipynb").read_text())["cells"] if c["cell_type"] == "code"],
                          "helpers": (ROOT / "scripts" / "extraction_utils.py").read_text()})
pipeline["code_hash"] = code_hash
extraction_version = (f"{len(GENERATED)} model responses generated in this run; other evidence may be cached"
                      if GENERATED else "cached model evidence replay; no new model generation in this run")
meta = {"code_hash": code_hash, "run_date": RUN_DATE, "prompt_hash": PROMPT_HASH, "pipeline_hash": request_hash(pipeline),
        "judge_model": JUDGE["id"], "extraction_version": extraction_version}
res_ = res_.assign(**meta) if len(res_) else pd.DataFrame(columns=["date"])
final_cols = ["date", "date_recorded", "company", "direction", "ticker", "proposed_ticker", "quote", "n_models", "avg_confidence", "quote_score", "ticker_votes",
              "ticker_llm1", "ticker_llm2", "ticker_llm3", "is_call", "judge_reason", "judge_error", "match_error", "ticker_errors",
              "ticker_status", "ticker_name", "ticker_src", "ticker_override_source", "ticker_override_effective_date", "ticker_override_from", "ticker_override_to", "listing_status_at_episode", "security_master_scope", "security_master_hash", "ticker_lookup_hash", "models",
              "direction_consensus", "start_s", "title", "doc_id", "run_date", "prompt_hash", "pipeline_hash", "code_hash", "judge_model", "extraction_version"]
res_ = res_.reindex(columns=final_cols)
if len(res_): res_["quote"] = res_.quote.fillna("").map(lambda q: " ".join(q.split()[:40]))
final = res_.set_index("date")
unresolved = res_[res_.ticker.isna()]
cands.drop(columns=["items"]).to_parquet(OUT / "trade_recs_candidates.parquet")
unresolved.to_csv(OUT / "trade_recs_unresolved.csv", index=False)
final.to_parquet(OUT / "trade_recs_final.parquet"); final.to_csv(OUT / "trade_recs_final.csv")
print(f"{len(cands)} candidates -> {len(final)} judge-approved calls retained ({len(unresolved)} require ticker review)")

## Cost

In [ ]:
cost = pd.DataFrame(COSTS, columns=["stage", "model", "cost"]).groupby(["stage", "model"]).agg(calls=("cost", "size"), cost=("cost", "sum"))
print(cost.round(5).to_string())
print(f"\ntotal for everything above: ${cost.cost.sum():.4f}")